# Lenguajes y Autómatas II (SCD-1016)
## Sesión 24 · Control de Flujo: Sentencias If e If-Else

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo ejecuta el programa una rama y evita la otra si solo sabe avanzar instrucción por instrucción?
>
> **Objetivo de la sesión:** Traducir una condición booleana a saltos condicionales con cortocircuito, usando una etiqueta de salida común.

**Entregable:** tu función emitir_condicion() en artesano_sesion24_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion24.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes dos operandos, un operador lógico (& para AND, | para OR) y una etiqueta de salida. Debes escribir los saltos que evalúan el primer operando y, solo si hace falta, el segundo.

Al final de la sesión tendrás: **Una función en C que emite el esqueleto de saltos de una condición compuesta y respeta el cortocircuito.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 24](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion24.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion24.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion24.h"

int emitir_condicion(char lineas[][64], int max, const char *a, const char *b, char op, const char *etiqueta);
```

- Para & usa IFFALSE: si el operando es falso, se salta a la etiqueta de salida.
- Para | usa IFTRUE: si el operando es verdadero, se salta a la etiqueta de salida.
- Si el segundo operando es NULL, emite un solo salto; no inventes una comparación.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion24_c.c` junto con `artesano_sesion24_c.c`
> (o `ia_sesion24_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `a && b (primer salto)` | `IFFALSE a GOTO L9` |
| 2 | `a && b (segundo salto)` | `IFFALSE b GOTO L9` |
| 3 | `a && b (dos saltos)` | 2 |
| 4 | `a || b (primer salto)` | `IFTRUE a GOTO L9` |
| 5 | `a || b (segundo salto)` | `IFTRUE b GOTO L9` |
| 6 | `condicion simple (b NULL)` | `IFFALSE a GOTO L0` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 24 - Saltos condicionales con cortocircuito ===
Caso                               esperado:              obtenido:              resultado
[ 1] a && b (primer salto)
     esperado: IFFALSE a GOTO L9
     obtenido: IFFALSE a GOTO L9
     -> OK
[ 2] a && b (segundo salto)
     esperado: IFFALSE b GOTO L9
     obtenido: IFFALSE b GOTO L9
     -> OK
[ 3] a && b (dos saltos)                esperado: 2            obtenido: 2            OK
[ 4] a || b (primer salto)
     esperado: IFTRUE a GOTO L9
     obtenido: IFTRUE a GOTO L9
     -> OK
[ 5] a || b (segundo salto)
     esperado: IFTRUE b GOTO L9
     obtenido: IFTRUE b GOTO L9
     -> OK
[ 6] condicion simple (b NULL)
     esperado: IFFALSE a GOTO L0
     obtenido: IFFALSE a GOTO L0
     -> OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion24.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] AND emite IFFALSE y OR emite IFTRUE, siempre hacia la etiqueta recibida.
- [ ] El segundo operando se evalúa con un salto propio y no se mezcla con el primero.
- [ ] Una condición sin segundo operando emite una sola instrucción y no se cae.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion24_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion24.h
/* api_sesion24.h - contrato de la Sesion 24
 * Saltos condicionales con cortocircuito
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION24_H
#define API_SESION24_H

int emitir_condicion(char lineas[][64], int max, const char *a, const char *b, char op, const char *etiqueta);

#endif /* API_SESION24_H */


In [ ]:
%%writefile artesano_sesion24_c.c
/* artesano_sesion24_c.c - Fase 1: El Artesano (sin IA)
 * Saltos condicionales con cortocircuito.
 *
 * Implementa lo que declara api_sesion24.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion24.h"
#include <stdio.h>
#include <string.h>

int emitir_condicion(char lineas[][64], int max, const char *a, const char *b, char op, const char *etiqueta) {
          (void)lineas; (void)max; (void)a; (void)b; (void)op; (void)etiqueta; /* TODO: saltos con cortocircuito */
          return 0;
      }


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion24_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion24_c.c
/* ia_sesion24_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion24.h. No puede traer main().
 */
#include "api_sesion24.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion24_c.c
/* harness_sesion24_c.c - auditoria automatica de la Sesion 24
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion24_c.c artesano_sesion24_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion24.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 24 - Saltos condicionales con cortocircuito ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { char l[8][64] = {{0}}; emitir_condicion(l, 8, "a", "b", '&', "L9"); verificar_str_largo("a && b (primer salto)", "IFFALSE a GOTO L9", l[0]); }
    { char l[8][64] = {{0}}; emitir_condicion(l, 8, "a", "b", '&', "L9"); verificar_str_largo("a && b (segundo salto)", "IFFALSE b GOTO L9", l[1]); }
    { char l[8][64] = {{0}}; int n = emitir_condicion(l, 8, "a", "b", '&', "L9"); verificar_int("a && b (dos saltos)", (long)(2), (long)(n)); }
    { char l[8][64] = {{0}}; emitir_condicion(l, 8, "a", "b", '|', "L9"); verificar_str_largo("a || b (primer salto)", "IFTRUE a GOTO L9", l[0]); }
    { char l[8][64] = {{0}}; emitir_condicion(l, 8, "a", "b", '|', "L9"); verificar_str_largo("a || b (segundo salto)", "IFTRUE b GOTO L9", l[1]); }
    { char l[4][64] = {{0}}; emitir_condicion(l, 4, "a", NULL, '&', "L0"); verificar_str_largo("condicion simple (b NULL)", "IFFALSE a GOTO L0", l[0]); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion24.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion24_c.c artesano_sesion24_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion24_c.c ia_sesion24_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion24.md` a `auditorias/auditoria_sesion24.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
